In [1]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("03_embeddings.ipynb")))

# Cargar descripciones
df_productos = pd.read_csv(os.path.join(BASE_DIR, "data", "processed", "productos_descripciones.csv"))
df_ratings = pd.read_csv(os.path.join(BASE_DIR, "data", "processed", "ratings_filtrado.csv"))

print(f"Productos: {len(df_productos):,}")
print(f"Ratings: {len(df_ratings):,}")
print(df_productos.head())

c:\Users\diego\OneDrive\Desktop\motor recomendaciones personalizado\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Productos: 57,289
Ratings: 394,908
    productId                                        descripcion
0  1304351475       hydrating lip balm with SPF 15 and vitamin E
1  1304482685        anti-aging serum with retinol and vitamin C
2  1403790965  moisturizing face cream with SPF 30 for daily use
3  1412759676  clarifying face wash for oily and acne-prone skin
4  3227001381  natural mineral foundation with buildable cove...


In [2]:
# Cargar modelo de embeddings
print("Cargando modelo de sentence-transformers...")
modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")
print("✓ Modelo cargado")

# Generar embeddings para todas las descripciones
# Esto puede tardar 2-3 minutos
print("\nGenerando embeddings...")
descripciones = df_productos["descripcion"].tolist()
embeddings = modelo_emb.encode(
    descripciones, 
    batch_size=256,
    show_progress_bar=True
)

print(f"\n✓ Embeddings generados")
print(f"Forma: {embeddings.shape}")
print(f"Cada producto = vector de {embeddings.shape[1]} dimensiones")

Cargando modelo de sentence-transformers...


c:\Users\diego\OneDrive\Desktop\motor recomendaciones personalizado\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diego\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 

✓ Modelo cargado

Generando embeddings...


Batches: 100%|██████████| 224/224 [04:19<00:00,  1.16s/it]



✓ Embeddings generados
Forma: (57289, 384)
Cada producto = vector de 384 dimensiones


In [3]:
import pickle

# Guardar embeddings
np.save(
    os.path.join(BASE_DIR, "data", "processed", "embeddings_productos.npy"),
    embeddings
)

# Guardar el dataframe de productos con su índice
df_productos.to_csv(
    os.path.join(BASE_DIR, "data", "processed", "productos_descripciones.csv"),
    index=False
)

print(f"✓ Embeddings guardados: {embeddings.shape}")
print(f"✓ Tamaño en disco: {embeddings.nbytes / 1024 / 1024:.1f} MB")

✓ Embeddings guardados: (57289, 384)
✓ Tamaño en disco: 83.9 MB


In [4]:
# Función de recomendación por embeddings
def recomendar_similares(product_id, n=10):
    # Buscar el producto en el dataframe
    if product_id not in df_productos["productId"].values:
        print(f"Producto {product_id} no encontrado")
        return
    
    # Obtener el índice y embedding del producto
    idx = df_productos[df_productos["productId"] == product_id].index[0]
    embedding_consulta = embeddings[idx].reshape(1, -1)
    
    # Calcular similitud con todos los productos
    similitudes = cosine_similarity(embedding_consulta, embeddings)[0]
    
    # Obtener los N más similares (excluyendo el mismo producto)
    indices_similares = similitudes.argsort()[::-1][1:n+1]
    
    print(f"Producto consultado: {product_id}")
    print(f"Descripción: {df_productos.iloc[idx]['descripcion']}")
    print(f"\nProductos similares:\n")
    
    for i, idx_similar in enumerate(indices_similares, 1):
        producto = df_productos.iloc[idx_similar]
        score = similitudes[idx_similar]
        print(f"{i}. {producto['productId']} | Score: {score:.4f}")
        print(f"   {producto['descripcion']}")

# Probar con un producto popular
producto_prueba = "B0043OYFKU"
recomendar_similares(producto_prueba)

Producto consultado: B0043OYFKU
Descripción: hydrating lip balm with SPF 15 and vitamin E

Productos similares:

1. 1304351475 | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
2. B00CC5AF6A | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
3. B00CBT6UJI | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
4. B00CBQVWGW | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
5. B00CBMLI8S | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
6. B00CBI16K2 | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
7. B00CBCGY3W | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
8. B00CA05FO4 | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
9. B00C8YCFU4 | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E
10. B00C89Z5KG | Score: 1.0000
   hydrating lip balm with SPF 15 and vitamin E


In [6]:
import random
random.seed(42)

categorias = [
    "face cream", "serum", "shampoo", "lipstick", "foundation",
    "face scrub", "conditioner", "mascara", "eye cream", "toner",
    "lip balm", "face wash", "hair mask", "setting powder", "face oil",
    "blush", "highlighter", "primer", "concealer", "nail polish",
    "body lotion", "sunscreen", "cleansing oil", "micellar water", "bb cream"
]

ingredientes = [
    "with vitamin C", "with retinol", "with hyaluronic acid",
    "with argan oil", "with SPF 30", "with niacinamide",
    "with keratin", "with rose hip oil", "with collagen",
    "with green tea extract", "with salicylic acid", "with vitamin E",
    "with caffeine", "with peptides", "with aloe vera",
    "with coconut oil", "with charcoal", "with glycerin"
]

beneficios = [
    "for dry skin", "for oily skin", "for sensitive skin",
    "for anti-aging", "for brightening", "for hydration",
    "for acne-prone skin", "for damaged hair", "for fine hair",
    "for daily use", "for long-lasting wear", "for deep nourishment",
    "for combination skin", "for mature skin", "for all skin types",
    "for dark spots", "for pore minimizing", "for radiant glow"
]

intensidades = [
    "lightweight", "intensive", "gentle", "ultra-hydrating",
    "fast-absorbing", "long-lasting", "natural", "professional"
]

productos_unicos = df_productos["productId"].unique()
descripciones_unicas = {
    producto: f"{random.choice(intensidades)} {random.choice(categorias)} {random.choice(ingredientes)} {random.choice(beneficios)}"
    for producto in productos_unicos
}

df_productos["descripcion"] = df_productos["productId"].map(descripciones_unicas)

print(f"Descripciones únicas: {df_productos['descripcion'].nunique():,} de {len(df_productos):,} productos")
print("\nEjemplos:")
print(df_productos["descripcion"].head(10).to_string())

Descripciones únicas: 38,161 de 57,289 productos

Ejemplos:
0    intensive face cream with collagen for damaged...
1    ultra-hydrating foundation with argan oil for ...
2      intensive concealer with peptides for oily skin
3    lightweight shampoo with keratin for damaged hair
4     lightweight primer with keratin for radiant glow
5         natural mascara with aloe vera for fine hair
6    lightweight bb cream with niacinamide for matu...
7    long-lasting eye cream with SPF 30 for acne-pr...
8    long-lasting lipstick with hyaluronic acid for...
9     intensive face wash with vitamin E for fine hair


In [7]:
print("Regenerando embeddings con descripciones mejoradas...")
descripciones = df_productos["descripcion"].tolist()
embeddings = modelo_emb.encode(
    descripciones,
    batch_size=256,
    show_progress_bar=True
)

# Guardar
np.save(
    os.path.join(BASE_DIR, "data", "processed", "embeddings_productos.npy"),
    embeddings
)
df_productos.to_csv(
    os.path.join(BASE_DIR, "data", "processed", "productos_descripciones.csv"),
    index=False
)

print(f"✓ Embeddings regenerados: {embeddings.shape}")

Regenerando embeddings con descripciones mejoradas...


Batches: 100%|██████████| 224/224 [05:44<00:00,  1.54s/it]


✓ Embeddings regenerados: (57289, 384)


In [8]:
recomendar_similares("B0043OYFKU")

Producto consultado: B0043OYFKU
Descripción: professional bb cream with SPF 30 for mature skin

Productos similares:

1. B000JMAZQS | Score: 0.9751
   natural bb cream with SPF 30 for mature skin
2. B004D6PSI6 | Score: 0.9700
   intensive bb cream with SPF 30 for mature skin
3. B00007FCPX | Score: 0.9672
   gentle bb cream with SPF 30 for mature skin
4. B001AZSUV0 | Score: 0.9601
   lightweight bb cream with SPF 30 for mature skin
5. B009CNCIVK | Score: 0.9488
   professional bb cream with SPF 30 for combination skin
6. B0038AA4S2 | Score: 0.9357
   professional bb cream with SPF 30 for sensitive skin
7. B004IS92GI | Score: 0.9332
   intensive bb cream with SPF 30 for all skin types
8. B000KGTJGK | Score: 0.9332
   intensive bb cream with SPF 30 for all skin types
9. B004LJ0ZK6 | Score: 0.9332
   intensive bb cream with SPF 30 for all skin types
10. B00BCOJ2Q6 | Score: 0.9324
   gentle bb cream with SPF 30 for all skin types
